# Local Test — Silver + Product Classification

Runs entirely on your machine with pandas + sentence-transformers.  
No Spark, no Databricks cluster needed.

**Setup:**
```
pip install pandas sentence-transformers requests
```

**Before running:** point `LOCAL_DATA_DIR` to a folder containing:
```
YYYY-MM-DD/
  productos.csv
  sucursales.csv
  comercio.csv
```
You can get these by running `downloader.py` locally (change `VOLUME_BASE` to a local path).

In [ ]:
import pandas as pd
import numpy as np
import os
import json
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
LOCAL_DATA_DIR = Path(r"C:\Users\2371180\OneDrive - Cognizant\Documents\sepa_raw")

# Rows to keep in silver after neighborhood filtering.
# Set to None to keep everything.
SILVER_SAMPLE = 10_000

TARGET_BARRIOS   = ["monserrat", "once", "balvanera"]
TARGET_LOCALIDAD = ["ciudad autónoma de buenos aires",
                    "ciudad autonoma de buenos aires",
                    "capital federal"]

print(f"Config OK — silver sample: {SILVER_SAMPLE:,} rows")

In [ ]:
# ── Load CSVs ────────────────────────────────────────────────────────────────
# Always loads the most recent date folder only (fast for local testing).
# No row limit here — sampling happens after the silver filter.

def load_all_dates(base_dir: Path):
    date_dirs = sorted([d for d in base_dir.iterdir() if d.is_dir()])
    if not date_dirs:
        raise FileNotFoundError(f"No date subfolders found in {base_dir}")

    # Most recent date only for local testing
    date_dir  = date_dirs[-1]
    week_date = date_dir.name
    print(f"Loading {week_date} ...")

    dfs = {}
    for fname in ["productos.csv", "sucursales.csv", "comercio.csv"]:
        fpath = date_dir / fname
        if not fpath.exists():
            raise FileNotFoundError(f"Missing {fpath}")
        df = pd.read_csv(fpath, sep="|", dtype=str, encoding="utf-8")
        df["week_date"] = week_date
        dfs[fname] = df
        print(f"  {fname:20s} {len(df):>10,} rows")

    return dfs["productos.csv"], dfs["sucursales.csv"], dfs["comercio.csv"]

df_productos, df_sucursales, df_comercio = load_all_dates(LOCAL_DATA_DIR)

In [ ]:
# Quick peek at column names — useful to verify the real SEPA schema
print("productos  columns:", df_productos.columns.tolist())
print("sucursales columns:", df_sucursales.columns.tolist())
print("comercio   columns:", df_comercio.columns.tolist())

In [ ]:
# ── Silver equivalent (pandas) ───────────────────────────────────────────────
# Mirrors etl_superapp.py: filter neighborhood, join, normalize price.

# 1. Normalize neighborhood columns to lowercase
df_sucursales["barrio_lower"]    = df_sucursales["sucursales_barrio"].str.lower().str.strip()
df_sucursales["localidad_lower"] = df_sucursales["sucursales_localidad"].str.lower().str.strip()

# 2. Filter to target neighborhoods
mask = (
    df_sucursales["barrio_lower"].isin(TARGET_BARRIOS) |
    df_sucursales["localidad_lower"].isin(TARGET_LOCALIDAD)
)
local_sucursales = df_sucursales[mask].copy()
print(f"Branches in target area: {len(local_sucursales):,}")
print(local_sucursales["sucursales_barrio"].value_counts().head(10))

In [ ]:
# 3. Deduplicate comercio
df_comercio_dedup = df_comercio.drop_duplicates(subset=["id_comercio"])[["id_comercio", "comercio_bandera_nombre"]]

# 4. Join sucursales → comercio
sucursales_with_chain = local_sucursales.merge(df_comercio_dedup, on="id_comercio", how="left")

# 5. Keep best sucursal per (comercio, barrio, week) = most products
product_counts = (
    df_productos.groupby(["id_sucursal", "week_date"])["id_producto"]
    .count().reset_index(name="product_count")
)
sucursales_with_counts = sucursales_with_chain.merge(product_counts, on=["id_sucursal", "week_date"], how="inner")
sucursales_with_counts["rank"] = (
    sucursales_with_counts
    .groupby(["id_comercio", "barrio_lower", "week_date"])["product_count"]
    .rank(method="first", ascending=False)
)
best_sucursales = sucursales_with_counts[sucursales_with_counts["rank"] == 1].copy()

# 6. Join products onto best sucursales
silver = df_productos.merge(
    best_sucursales[["id_sucursal", "week_date", "id_comercio",
                     "sucursales_barrio", "sucursales_nombre",
                     "sucursales_calle", "sucursales_numero",
                     "comercio_bandera_nombre"]],
    on=["id_sucursal", "week_date"], how="inner"
)

# 7. Normalize price
silver["precio"] = (
    silver["productos_precio_lista"]
    .str.replace(",", ".", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)
silver = silver[silver["precio"] > 0].copy()

# 8. Rename columns
silver = silver.rename(columns={
    "productos_descripcion":                "producto",
    "productos_unidad_medida_presentacion": "presentacion",
    "comercio_bandera_nombre":              "cadena",
    "sucursales_nombre":                    "sucursal_nombre",
    "sucursales_barrio":                    "barrio",
})

# 9. Sample to SILVER_SAMPLE rows (random, reproducible)
if SILVER_SAMPLE and len(silver) > SILVER_SAMPLE:
    silver = silver.sample(SILVER_SAMPLE, random_state=42).reset_index(drop=True)
    print(f"Sampled to {SILVER_SAMPLE:,} rows")

print(f"Silver rows    : {len(silver):,}")
print(f"Unique products: {silver['id_producto'].nunique():,}")
print(f"Unique chains  : {silver['cadena'].nunique():,}")
silver[["week_date", "producto", "cadena", "barrio", "precio"]].head(10)

In [ ]:
# ── Unique products for classification ───────────────────────────────────────
unique_products = silver[["id_producto", "producto"]].drop_duplicates(subset=["id_producto"])
print(f"Unique products to classify: {len(unique_products):,}")
unique_products.head(10)

In [ ]:
# ── Step 1: Keyword rules (same as notebook 02) ───────────────────────────────
# Quick first pass — classifies ~20-30% immediately.

CATEGORY_KEYWORDS = {
    "arroz":       ["arroz"],
    "aceite":      ["aceite"],
    "fideos":      ["fideos", "pasta ", "tallar", "spaghetti"],
    "leche":       ["leche"],
    "yerba":       ["yerba"],
    "azucar":      ["azucar", "azúcar"],
    "jabon":       ["jabon", "jabón", "detergente"],
    "galletitas":  ["galleta", "galletita"],
    "pan":         ["pan lactal", "pan de mi", "panificad"],
    "cafe":        ["cafe ", "café", "nescafe"],
    "te":          ["te en saq", "te herbal", "infusion"],
    "gaseosa":     ["gaseosa", "coca", "pepsi", "sprite", "fanta"],
    "agua":        ["agua mineral", "agua sin gas", "agua con gas"],
    "jugo":        ["jugo", "zumo"],
    "cerveza":     ["cerveza"],
    "vino":        ["vino "],
    "manteca":     ["manteca", "margarina"],
    "queso":       ["queso"],
    "yogur":       ["yogur", "yoghurt"],
    "harina":      ["harina"],
    "shampoo":     ["shampoo", "champú", "champu"],
    "papel":       ["papel higienico", "papel cocina", "servilleta"],
    "desodorante": ["desodorante"],
    "chocolate":   ["chocolate"],
    "mermelada":   ["mermelada", "dulce de"],
    "mayonesa":    ["mayonesa"],
}

def apply_keyword_rules(name: str) -> str | None:
    if not name:
        return None
    lower = name.lower()
    for cat, keywords in CATEGORY_KEYWORDS.items():
        if any(kw in lower for kw in keywords):
            return cat
    return None

unique_products["categoria_regla"] = unique_products["producto"].apply(apply_keyword_rules)

classified_kw   = unique_products[unique_products["categoria_regla"].notna()]
unclassified_kw = unique_products[unique_products["categoria_regla"].isna()]

print(f"Classified by keyword rules : {len(classified_kw):,} ({len(classified_kw)*100//len(unique_products)}%)")
print(f"Still unclassified          : {len(unclassified_kw):,}")
print()
print(classified_kw["categoria_regla"].value_counts().to_string())

In [ ]:
# ── Step 2: Build embeddings + centroids from keyword-classified products ─────
# This is the local equivalent of notebook 05.

from sentence_transformers import SentenceTransformer

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print(f"Embedding {len(classified_kw):,} keyword-classified products to build centroids...")
labeled_embs = model.encode(
    classified_kw["producto"].tolist(),
    show_progress_bar=True, batch_size=128
)
classified_kw = classified_kw.copy()
classified_kw["embedding"] = list(labeled_embs)

# Compute per-category centroid (normalized)
centroids = {}
for cat, group in classified_kw.groupby("categoria_regla"):
    vecs = np.array(group["embedding"].tolist())
    c = vecs.mean(axis=0)
    norm = np.linalg.norm(c)
    centroids[cat] = c / norm if norm > 1e-10 else c

centroid_matrix = np.array(list(centroids.values()))  # (n_cats, dim)
centroid_names  = list(centroids.keys())

print(f"Centroids built for {len(centroids)} categories.")

In [ ]:
# ── Step 3: Classify unclassified products by centroid similarity ─────────────
# Local equivalent of notebook 06 (auto-assign only, no LLM call here).

CONFIDENCE_AUTO = 0.82  # slightly lower than Databricks since centroids are smaller
CONFIDENCE_LLM  = 0.60

print(f"Embedding {len(unclassified_kw):,} unclassified products...")
unc_embs = model.encode(
    unclassified_kw["producto"].tolist(),
    show_progress_bar=True, batch_size=128
)

# Cosine similarity vs all centroids
norms    = np.linalg.norm(unc_embs, axis=1, keepdims=True)
normed   = unc_embs / (norms + 1e-10)
sims     = normed @ centroid_matrix.T   # (n_unclassified, n_cats)
top_idx  = sims.argmax(axis=1)
top_sims = sims.max(axis=1)

unclassified_kw = unclassified_kw.copy()
unclassified_kw["top_categoria"] = [centroid_names[i] for i in top_idx]
unclassified_kw["similitud"]     = top_sims.round(4)
unclassified_kw["route"] = np.where(
    top_sims >= CONFIDENCE_AUTO, "auto",
    np.where(top_sims >= CONFIDENCE_LLM, "llm_confirm", "llm_new_cat")
)

print("\nRouting summary:")
print(unclassified_kw["route"].value_counts().to_string())
print(f"\nAuto-assigned: {(unclassified_kw['route']=='auto').sum():,}")
print(f"LLM confirm  : {(unclassified_kw['route']=='llm_confirm').sum():,}")
print(f"LLM new cat  : {(unclassified_kw['route']=='llm_new_cat').sum():,}")

In [ ]:
# ── Step 4: Inspect auto-assigned ────────────────────────────────────────────
auto = unclassified_kw[unclassified_kw["route"] == "auto"].copy()
print(f"Auto-assigned: {len(auto):,}")
print()
print(auto.groupby("top_categoria").size().sort_values(ascending=False).to_string())
print()
# Sample per category
for cat, grp in auto.groupby("top_categoria"):
    samples = grp.sort_values("similitud", ascending=False).head(3)
    print(f"  {cat}")
    for _, r in samples.iterrows():
        print(f"    [{r['similitud']:.2f}] {r['producto']}")

In [ ]:
# ── Step 5: Inspect review queue (uncertain items) ────────────────────────────
queue = unclassified_kw[unclassified_kw["route"] != "auto"].copy()
queue_summary = (
    queue.groupby(["top_categoria", "route"])
    .agg(count=("id_producto", "count"),
         examples=("producto", lambda x: list(x.head(4))))
    .sort_values("count", ascending=False)
)
print(f"Review queue: {len(queue):,} items\n")
for (cat, route), row in queue_summary.head(30).iterrows():
    print(f"  [{route:12s}] {cat:25s}  ({row['count']} products)")
    for ex in row['examples']:
        print(f"    • {ex}")
    print()

In [ ]:
# ── Step 6: Merge all classifications into final table ────────────────────────
# Combines: keyword rules + auto-assigned embeddings + review queue

kw_classified = classified_kw[["id_producto", "producto"]].copy()
kw_classified["categoria"]  = classified_kw["categoria_regla"].values
kw_classified["confianza"]   = 0.80
kw_classified["metodo"]      = "regla"

emb_auto = auto[["id_producto", "producto"]].copy()
emb_auto["categoria"] = auto["top_categoria"].values
emb_auto["confianza"]  = auto["similitud"].values
emb_auto["metodo"]     = "embedding_auto"

review = queue[["id_producto", "producto"]].copy()
review["categoria"] = queue["top_categoria"].values   # best guess
review["confianza"]  = queue["similitud"].values
review["metodo"]     = "pendiente_revision"

gold_productos_categorias = pd.concat([kw_classified, emb_auto, review], ignore_index=True)

print(f"Total products classified: {len(gold_productos_categorias):,}")
print()
print(gold_productos_categorias["metodo"].value_counts().to_string())
print()
print(gold_productos_categorias["categoria"].value_counts().head(20).to_string())

In [ ]:
# ── Step 7: Join silver + categories = comparable price table ─────────────────
silver_classified = silver.merge(
    gold_productos_categorias[["id_producto", "categoria", "confianza", "metodo"]],
    on="id_producto", how="left"
)

# Sample: compare prices across brands within the same category
cat_to_inspect = "aceite"   # change this to any category

comparison = (
    silver_classified[silver_classified["categoria"] == cat_to_inspect]
    .groupby(["week_date", "cadena", "producto"])["precio"]
    .mean().round(2).reset_index()
    .sort_values(["week_date", "precio"])
)
print(f"Price comparison for '{cat_to_inspect}':")
print(comparison.to_string(index=False))

In [ ]:
# ── Save results locally (optional) ──────────────────────────────────────────
out_dir = LOCAL_DATA_DIR.parent / "test_output"
out_dir.mkdir(exist_ok=True)

silver.to_parquet(out_dir / "silver_prices.parquet", index=False)
gold_productos_categorias.to_parquet(out_dir / "gold_productos_categorias.parquet", index=False)
silver_classified.to_parquet(out_dir / "silver_classified.parquet", index=False)

print(f"Saved to {out_dir}")